# 01 Smoke run: baseline line, one replication

T3.1.5 smoke check. **Simulated, benchmark-parameterized line; not plant data.** Written at T3.1.5 to show the line mechanics: breakdowns, minor stops, shift calendar, blocking and starvation. Changeovers, quality, staffing and QA were added in T3.2.x and now appear in the run too. The state table below uses the raw log, before the calendar overlay; for OEE use `ffdt.metrics` (docs/metrics_schema.md).

In [ ]:
import pandas as pd
import plotly.graph_objects as go

from ffdt.config import load_config
from ffdt.sim.line import Line
from ffdt.viz import SIM_LABEL, STATE_COLORS, STATE_ORDER

cfg = load_config("../configs/baseline_line.yaml")
line = Line(cfg, root_seed=1, replication=0)
completed = line.run()
print(
    f"completed={completed}  campaign={line.campaign_end_min / 60:.1f} h  packed={line.ledger.packed:,} vials"
)

In [ ]:
states = pd.DataFrame(line.log.states, columns=["station", "unit", "state", "start_min", "end_min"])
states["minutes"] = states.end_min - states.start_min
table = states.pivot_table(
    index="station", columns="state", values="minutes", aggfunc="sum", fill_value=0
)
table = table.reindex(
    index=[s.name for s in cfg.stations], columns=[s for s in STATE_ORDER if s in table.columns]
)
(table / table.sum(axis=1).values[:, None] * 100).round(
    1
)  # % of campaign time per station (table view)

In [ ]:
# State timeline for the first production day (two shifts)
day = states[states.start_min < 1440].copy()
day["end_min"] = day.end_min.clip(upper=1440)
order = [s.name for s in cfg.stations]
fig = go.Figure()
for st in STATE_ORDER:
    d = day[day.state == st]
    if d.empty:
        continue
    fig.add_bar(
        y=d.station,
        x=(d.end_min - d.start_min) / 60,
        base=d.start_min / 60,
        orientation="h",
        name=st,
        marker=dict(color=STATE_COLORS[st], line=dict(width=0)),
        hovertemplate="%{y}<br>" + st + "<br>%{base:.2f} h, %{x:.2f} h<extra></extra>",
    )
fig.update_layout(
    barmode="overlay",
    title=f"Station states, day 1 {SIM_LABEL}",
    xaxis_title="Hours from campaign start",
    yaxis=dict(categoryorder="array", categoryarray=order[::-1]),
    legend_title="State",
    height=380,
    plot_bgcolor="#fcfcfb",
    xaxis=dict(gridcolor="#e7e6e1"),
)
fig.show()